# 1. Possible games

A player's **possible games** are the games his team played while he was on the roster, from his NBA debut on. Every game he missed counts: injury, rest or suspension.

This notebook works them out for every player in every season, and checks them against well-known cases.

In [1]:
import pandas as pd

from xppg import data, possible
from xppg.raw import PROJECT_ROOT
from xppg.show import table

d = PROJECT_ROOT / "data" / "processed"
P, T, R, M = (pd.read_parquet(d / f"{n}.parquet") for n in ["player_games", "team_games", "rosters", "player_movement"])
debut = pd.read_parquet(d / "debuts.parquet").set_index("player_id").debut.to_dict()

stints = possible.build_stints(P, T, R, M, debut)   # checks: never more games played than possible
ps = possible.player_seasons(stints, P, R)
ps.to_parquet(d / "player_seasons.parquet", index=False)
team_abbr = T.drop_duplicates("team_id").set_index("team_id").team_abbreviation

def note(r):
    notes = []
    if r.two_way: notes.append("Two-way contract")
    if r.ten_day: notes.append("10-day contract")
    if r.undated: notes.append("Some dates unknown")
    return ", ".join(notes)
ps["Note"] = ps.apply(note, axis=1)

## 1. Some well-known cases

**What you're looking at:** games played and possible games for players whose seasons were unusual: missed whole seasons, traded while injured, or waived.  
**Why it matters:** if these look right, the method can be trusted for everyone else.

In [2]:
cases = [("Joel Embiid", "2023-24", "Played 39 of the 76ers' 82 games"),
         ("Klay Thompson", "2020-21", "Missed the whole season (torn Achilles)"),
         ("Ben Simmons", "2021-22", "Did not play; traded from Philadelphia to Brooklyn in February"),
         ("Kevin Durant", "2022-23", "Injured when traded from Brooklyn to Phoenix in February"),
         ("James Harden", "2021-22", "Traded from Brooklyn to Philadelphia in February"),
         ("James Wiseman", "2024-25", "Injured in his first game; traded and waived in February")]
rows = []
for name, season, story in cases:
    r = ps[(ps.player_name == name) & (ps.season == season)].iloc[0]
    teams = " then ".join(team_abbr[t] for t in stints[(stints.player_id == r.player_id) & (stints.season == season)]
                          .sort_values("start").team_id)
    rows.append((name, season, teams, r.gp, r.possible_games, story))
examples = pd.DataFrame(rows, columns=["Player", "Season", "Team(s)", "Played", "Possible games", "What happened"])
table(examples, "Possible games: well-known cases")

Player,Season,Team(s),Played,Possible games,What happened
Joel Embiid,2023-24,PHI,39,82,Played 39 of the 76ers' 82 games
Klay Thompson,2020-21,GSW,0,72,Missed the whole season (torn Achilles)
Ben Simmons,2021-22,PHI then BKN,0,82,Did not play; traded from Philadelphia to Brooklyn in February
Kevin Durant,2022-23,BKN then PHX,47,80,Injured when traded from Brooklyn to Phoenix in February
James Harden,2021-22,BKN then PHI,65,82,Traded from Brooklyn to Philadelphia in February
James Wiseman,2024-25,IND,1,49,Injured in his first game; traded and waived in February


## 2. Possible games, season by season

**What you're looking at:** for each season, how many players are counted, and on average how many games they played against how many they could have played.  
**Why it matters:** the gap between the two averages is the playing time lost across the league, which points per game ignores.

In [3]:
summary = ps.groupby("season").agg(Players=("player_id", "size"), **{
    "Average games played": ("gp", "mean"),
    "Average possible games": ("possible_games", "mean"),
    "Players who missed the whole season": ("gp", lambda x: (x == 0).sum()),
})
summary.index = [s + (" (so far)" if s == data.IN_PROGRESS else "") for s in summary.index]
table(summary.rename_axis("Season"), "Games played against possible games")

,Players,Average games played,Average possible games,Players who missed the whole season
Season,,,,
2020-21,543,42.5,64.8,3
2021-22,615,42.3,67.1,10
2022-23,545,47.5,73.5,6
2023-24,579,45.6,73.1,7
2024-25,571,46.1,73.6,2
2025-26,586,45.5,71.7,4
2026-27 (so far),511,11.1,16.6,36


## 3. Players with a note

**What you're looking at:** how many player-seasons carry each kind of note.  
**Why it matters:** **two-way** players are limited by their contract in how many NBA games they can play, so some of their missed games are not about health. **10-day** contracts end 10 days after signing. **Some dates unknown** means NBA.com has no record of when the player joined or left, so his possible games stop at his last game. These players are kept in, and marked wherever they appear.

In [4]:
notes = pd.DataFrame({"Player-seasons": [ps.two_way.sum(), ps.ten_day.sum(), ps.undated.sum(), len(ps)]},
                     index=["Two-way contract", "10-day contract", "Some dates unknown", "All player-seasons"])
table(notes.rename_axis("Note"), "Player-seasons with a note, 2020-21 to 2026-27")

,Player-seasons
Note,
Two-way contract,576
10-day contract,193
Some dates unknown,17
All player-seasons,"3,950"


---
Every rule used here is listed, with its reason, in `docs/DECISIONS.md`.